# Day 14: Model Selection and Introduction to RAG

Welcome to Day 14! In this notebook, we'll explore:
- How to select the right LLM for your task
- Understanding model benchmarks and leaderboards
- Introduction to RAG (Retrieval Augmented Generation)
- Working with vector embeddings

Let's get started!


## Setup and Installation

First, let's import all the necessary libraries.


In [1]:
# Import required libraries
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import numpy as np

# Load environment variables from .env file
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✅ Setup complete!")


✅ Setup complete!


## Part 1: Understanding Model Selection

When choosing an LLM, we need to consider several factors:
- **Parameters**: Size of the model (bigger usually means smarter)
- **Context Window**: How much text the model can process at once
- **Cost**: API pricing or compute costs
- **Speed**: Response time and latency
- **Benchmarks**: Performance on standardized tests

Let's explore how different models perform!


## Part 2: Simple RAG Implementation

RAG stands for **Retrieval Augmented Generation**. It allows us to give LLMs access to external knowledge.

### The Problem
LLMs have knowledge cutoffs and don't know about:
- Your company's private data
- Recent events after their training
- Specific domain knowledge

### The Solution
Retrieve relevant information from a knowledge base and add it to the prompt!


### Step 1: Create a Simple Knowledge Base

Let's create a knowledge base about a fictional insurance company called "InsureElm".


In [2]:
# Create a simple knowledge base using a dictionary
knowledge = {
    "lancaster": "Avery Lancaster is the Co-founder and CEO of InsureElm, an innovative insurance technology company.",
    "thompson": "Maxine Thompson is a Senior Data Engineer at InsureElm, specializing in building data pipelines.",
    "rivera": "Jordan Rivera is the CTO at InsureElm, leading the engineering team.",
    "car": "CarElm is InsureElm's auto insurance product with rates starting at $89/month.",
    "health": "HealthElm provides comprehensive medical coverage starting at $299/month.",
    "life": "LifeElm offers term and whole life insurance policies starting at $25/month.",
    "claim": "ClaimElm is an AI-powered claims processing system that processes claims in under 24 hours.",
}

print("✅ Knowledge base created!")
print(f"📚 Contains information about {len(knowledge)} topics")


✅ Knowledge base created!
📚 Contains information about 7 topics


### Step 2: Create a Function to Retrieve Relevant Context


In [3]:
def get_relevant_context(message):
    """Search the knowledge base for relevant context."""
    # Remove punctuation and convert to lowercase
    cleaned = ''.join(c for c in message if c.isalpha() or c.isspace())
    words = cleaned.lower().split()
    
    # Look up each word - pythonic way with list comprehension
    return [knowledge[word] for word in words if word in knowledge]

# Test it
test_question = "Who is Lancaster?"
context = get_relevant_context(test_question)
print(f"Question: {test_question}")
print(f"Found: {context}")


Question: Who is Lancaster?
Found: ['Avery Lancaster is the Co-founder and CEO of InsureElm, an innovative insurance technology company.']


### Step 3: Create the Chat Function with Simple RAG


In [4]:
def chat_simple(message, history):
    """Chat function with simple dictionary-based RAG."""
    # Get relevant context
    context_list = get_relevant_context(message)
    
    # Format context
    if context_list:
        context_text = "\n\nRelevant context:\n" + "\n".join(context_list)
    else:
        context_text = "\n\nNo relevant context found."
    
    # Build system message
    system_msg = f"""You represent InsureElm insurance company.
Answer questions about employees and products briefly and accurately.
If you don't know, say so.{context_text}"""
    
    # Build messages list
    messages = [{"role": "system", "content": system_msg}]
    for user_msg, asst_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": asst_msg})
    messages.append({"role": "user", "content": message})
    
    # Call OpenAI
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )
    return response.choices[0].message.content

print("✅ Simple RAG chat function created!")


✅ Simple RAG chat function created!


### Step 4: Test Simple RAG


In [5]:
# Test questions
questions = [
    "Who is Lancaster?",  # Works
    "What is car?",  # Works
    "Who is Avery?",  # Fails - only last names indexed!
]

for q in questions:
    response = chat_simple(q, [])
    print(f"Q: {q}")
    print(f"A: {response}\n")


Q: Who is Lancaster?
A: Avery Lancaster is the Co-founder and CEO of InsureElm, an innovative insurance technology company.

Q: What is car?
A: A car is a wheeled motor vehicle used for transportation. It typically has four wheels, an internal combustion engine or electric motor, and is designed to carry passengers.

Q: Who is Avery?
A: I'm sorry, but I don't have any specific information about an employee named Avery at InsureElm.



## Part 3: Vector Embeddings - The Better Way

### What are Vector Embeddings?
- Convert text to numbers that represent **meaning**
- Similar meanings → similar numbers
- Enable semantic search (not just exact matching)

Let's see how this works!


### Step 5: Create Embeddings with OpenAI


In [6]:
def get_embedding(text, model="text-embedding-3-small"):
    """Get vector embedding for text using OpenAI."""
    text = text.replace("\n", " ")
    response = client.embeddings.create(input=[text], model=model)
    return response.data[0].embedding

# Test with similar and different texts
text1 = "Ticket prices to London"
text2 = "Flight costs to Heathrow Airport"
text3 = "I love pizza"

emb1 = get_embedding(text1)
emb2 = get_embedding(text2)
emb3 = get_embedding(text3)

print(f"Embedding length: {len(emb1)} numbers")
print(f"Text 1 first 5: {emb1[:5]}")
print(f"Text 2 first 5: {emb2[:5]}")
print(f"Text 3 first 5: {emb3[:5]}")


Embedding length: 1536 numbers
Text 1 first 5: [-0.012582375667989254, -0.04338422417640686, 0.036534082144498825, -0.02694864198565483, -0.03886503353714943]
Text 2 first 5: [-0.03407908231019974, -0.020937765017151833, 0.044260844588279724, -0.040859561413526535, -0.0655519887804985]
Text 3 first 5: [-0.008173821493983269, -0.04009635001420975, -0.062125395983457565, -0.01985226385295391, -0.005779359955340624]


### Step 6: Calculate Similarity


In [7]:
def cosine_similarity(v1, v2):
    """Calculate cosine similarity between two vectors."""
    v1, v2 = np.array(v1), np.array(v2)
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

# Compare similarities
sim_1_2 = cosine_similarity(emb1, emb2)
sim_1_3 = cosine_similarity(emb1, emb3)

print(f"'{text1}' vs '{text2}': {sim_1_2:.4f}")
print("  → Same topic (travel to London) - HIGH similarity!\n")

print(f"'{text1}' vs '{text3}': {sim_1_3:.4f}")
print("  → Different topics - LOW similarity!")


'Ticket prices to London' vs 'Flight costs to Heathrow Airport': 0.5881
  → Same topic (travel to London) - HIGH similarity!

'Ticket prices to London' vs 'I love pizza': 0.0169
  → Different topics - LOW similarity!


### Step 7: Create Embeddings for Knowledge Base


In [8]:
print("Creating embeddings for knowledge base...\n")

knowledge_embeddings = {}
for key, text in knowledge.items():
    print(f"Embedding: {key}...")
    knowledge_embeddings[key] = {
        "text": text,
        "embedding": get_embedding(text)
    }

print(f"\n✅ Created embeddings for {len(knowledge_embeddings)} entries!")


Creating embeddings for knowledge base...

Embedding: lancaster...
Embedding: thompson...
Embedding: rivera...
Embedding: car...
Embedding: health...
Embedding: life...
Embedding: claim...

✅ Created embeddings for 7 entries!


### Step 8: Semantic Search Function


In [9]:
def semantic_search(query, top_k=2):
    """Find most relevant knowledge using embeddings."""
    query_emb = get_embedding(query)
    
    # Calculate similarity with each entry
    similarities = []
    for key, data in knowledge_embeddings.items():
        sim = cosine_similarity(query_emb, data["embedding"])
        similarities.append((key, data["text"], sim))
    
    # Sort by similarity (highest first)
    similarities.sort(key=lambda x: x[2], reverse=True)
    return similarities[:top_k]

# Test semantic search - these failed with simple RAG!
test_queries = [
    "Who is the CEO?",  # No "CEO" in keys!
    "Tell me about auto insurance",  # "auto" not "car"!
    "Who is Avery?",  # Only first name!
]

for query in test_queries:
    print(f"\n🔍 Query: {query}")
    results = semantic_search(query, top_k=2)
    for i, (key, text, score) in enumerate(results, 1):
        print(f"  {i}. [{key}] Score: {score:.4f}")
        print(f"     {text[:60]}...")



🔍 Query: Who is the CEO?
  1. [rivera] Score: 0.3702
     Jordan Rivera is the CTO at InsureElm, leading the engineeri...
  2. [lancaster] Score: 0.3206
     Avery Lancaster is the Co-founder and CEO of InsureElm, an i...

🔍 Query: Tell me about auto insurance
  1. [car] Score: 0.4597
     CarElm is InsureElm's auto insurance product with rates star...
  2. [life] Score: 0.3261
     LifeElm offers term and whole life insurance policies starti...

🔍 Query: Who is Avery?
  1. [lancaster] Score: 0.4935
     Avery Lancaster is the Co-founder and CEO of InsureElm, an i...
  2. [rivera] Score: 0.1823
     Jordan Rivera is the CTO at InsureElm, leading the engineeri...


### Step 9: Improved Chat with Semantic RAG


In [10]:
def chat_semantic(message, history):
    """Improved chat with semantic search."""
    # Use semantic search
    results = semantic_search(message, top_k=2)
    
    # Format context
    context_text = "\n\nRelevant information:\n"
    for key, text, score in results:
        if score > 0.5:  # Only if similarity above threshold
            context_text += f"\n{text}\n"
    
    # Build system message
    system_msg = f"""You represent InsureElm insurance company.
Answer questions briefly and accurately.{context_text}"""
    
    # Build messages
    messages = [{"role": "system", "content": system_msg}]
    for user_msg, asst_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": asst_msg})
    messages.append({"role": "user", "content": message})
    
    # Call OpenAI
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )
    return response.choices[0].message.content

print("✅ Semantic RAG chat function created!")


✅ Semantic RAG chat function created!


### Step 10: Test Improved RAG


In [11]:
# Test questions that failed before!
improved_questions = [
    "Who is the CEO?",
    "Who is Avery?",
    "Tell me about auto insurance",
    "How do I file a claim?"
]

print("Testing improved semantic RAG:\n")
for q in improved_questions:
    response = chat_semantic(q, [])
    print(f"Q: {q}")
    print(f"A: {response}\n")


Testing improved semantic RAG:

Q: Who is the CEO?
A: I'm sorry, but I do not have specific information about the current CEO of InsureElm. Please check our official website or recent announcements for the most accurate information.

Q: Who is Avery?
A: I'm sorry, but I don't have specific information about individuals unless they are public figures or related to general knowledge. If you need help with insurance-related inquiries, feel free to ask!

Q: Tell me about auto insurance
A: Auto insurance is a policy purchased by vehicle owners to cover the costs associated with vehicle-related accidents, theft, and damage. It typically includes several types of coverage:

1. **Liability Coverage**: Covers damages to other vehicles or injuries to other people in an accident you cause.
2. **Collision Coverage**: Covers damage to your vehicle from a collision, regardless of who is at fault.
3. **Comprehensive Coverage**: Covers non-collision related incidents such as theft, vandalism, or natur

## Part 4: Interactive Demo with Gradio

Let's create interactive interfaces to compare both approaches!


### Simple RAG Interface


In [12]:
demo_simple = gr.ChatInterface(
    fn=chat_simple,
    title="InsureElm Assistant - Simple RAG (String Matching)",
    description="Uses exact string matching. Try: 'Who is Lancaster?' vs 'Who is Avery?'",
    examples=[
        "Who is Lancaster?",
        "What is car?",
        "Who is Avery?",  # This will fail!
    ]
)

demo_simple.launch(share=False)


c:\Users\HP\.conda\envs\llm-env\Lib\site-packages\gradio\chat_interface.py:339: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### Semantic RAG Interface


In [13]:
demo_semantic = gr.ChatInterface(
    fn=chat_semantic,
    title="InsureElm Assistant - Semantic RAG (Embeddings)",
    description="Uses vector embeddings for semantic search. Try the same questions!",
    examples=[
        "Who is the CEO?",
        "Who is Avery?",
        "Tell me about auto insurance",
        "How do I file a claim?",
    ]
)

demo_semantic.launch(share=False)


c:\Users\HP\.conda\envs\llm-env\Lib\site-packages\gradio\chat_interface.py:339: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Summary and Key Takeaways

### What We Learned:

1. **Model Selection**
   - No single "best" model - choose based on your task
   - Consider: parameters, cost, speed, context window
   - Use leaderboards: Artificial Analysis, Vellum, Live Bench

2. **Simple RAG**
   - Augment prompts with external knowledge
   - Dictionary lookup is simple but brittle
   - Fails with typos, synonyms, variations

3. **Vector Embeddings**
   - Convert text to numbers representing meaning
   - Similar meanings → similar vectors
   - Enable semantic search

4. **Semantic RAG**
   - Use embeddings to find relevant context
   - Works with synonyms and related concepts
   - Much more robust than string matching

### Two Types of LLMs:
- **Autoregressive** (GPT, Claude): Generate text
- **Encoder** (Embeddings): Convert text to vectors

### Next Steps:
- LangChain for production RAG
- Vector databases (Chroma, Pinecone)
- Advanced RAG techniques

Great job completing Day 14! 🎉
